# Cell and gene embeddings

This notebook extracts cell embeddings and time-resolved gene embeddings from the **released PerturbGen model used for the embedding analyses in the manuscript**: the masking model trained on the LPS data with the unstimulated `normal` state as the source and 90 min, 6 h and 10 h as targets. These embeddings are the input to the `gene_program_discovery` notebook.

The released checkpoint and the tokenized data it was trained on are downloaded from Hugging Face, so no training is needed. To train a model on your own data, see the `train_and_perturb` notebook.

## Download the released model and data

The checkpoint and its tokenized data are on Hugging Face (https://huggingface.co/lotfollahi-lab/PerturbGen). The checkpoint must be used with the tokenized data it was trained on, because the token IDs of the genes are defined by that tokenization.

In [ ]:
import os

from huggingface_hub import hf_hub_download, snapshot_download

import perturbgen as pg

REPO = "lotfollahi-lab/PerturbGen"
# masking checkpoint used for the manuscript's cell/gene-embedding analyses (normal source; targets 90 min, 6 h, 10 h)
MASKING_CKPT = hf_hub_download(
    REPO, "lps/embedding/20250714_2311_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_cosine_tp_1-2-3_s_0-epoch=07.ckpt"
)
# the tokenized data this checkpoint was trained on
LOCAL = snapshot_download(REPO, allow_patterns=["lps/tokenized/OURmed100m_LPS_all_tps_2k_hvg/*"])
TOKENIZED_DIR = os.path.join(LOCAL, "lps/tokenized/OURmed100m_LPS_all_tps_2k_hvg")
EMB_DIR = os.path.abspath("lps_cell_gene_embeddings")  # read by the gene_program_discovery notebook

## Build the model

The settings match the released checkpoint: `normal` source with 90 min, 6 h and 10 h as targets, time conditioning, the cosine masking scheduler, and the model dimensions the checkpoint was trained with (`max_len=797`, `tgt_vocab_size=2002`).

In [ ]:
model = pg.PerturbGen(
    TOKENIZED_DIR,
    source="normal",
    n_hvg=2000,
    n_layers=6, d_ff=32, d_model=768,
    pred_tps=[1, 2, 3],              # targets: 90 min, 6 h, 10 h
    var_list=["cell_type_harmonized", "cell_pairing_index", "time_after_LPS"],
    cond_list=["time_after_LPS"],    # time conditioning
    mask_scheduler="cosine",
    context_mode=True,
    max_len=797, tgt_vocab_size=2002,  # dimensions of the released checkpoint
)

## Extract cell and gene embeddings

Cell embeddings are stored in `obsm["cls_embeddings"]`; gene embeddings are extracted separately for each time point (`gene_embs_condition`) and stored in `varm`.

In [ ]:
EMB_DIR = model.get_embeddings(
    output_dir=EMB_DIR,
    masking_ckpt=MASKING_CKPT,
    return_cell_embs=True,
    return_gene_embs=True,
    gene_embs_condition="time_after_LPS",  # one set of gene embeddings per time point
)
print("embeddings at:", EMB_DIR)

Next: discover gene programs from these embeddings in the `gene_program_discovery` notebook.